In [3]:
%run 'utils/utils_helper_funcs.ipynb'
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Exception: File `'utils/utils_helper_funcs.ipynb'` not found.

In [1]:
def total_ozone_percent_cont(hist, other, reg_name, time_range=None):
    # Handle dates
    try:
        len(hist['time'])==len(other['time'])
    except:
        raise KeyError('Both datasets must have the same length time coordinate')
        
    if time_range != None:
        start = str(time_range[0])
        end = str(time_range[1])
        hist = hist.sel(time=slice(start, end))
        other = other.sel(time=slice(start, end))
    else:
            start = str((hist['time'].values)[0])[:4]
            end = str((hist['time'].values)[len(hist['time'])-1])[:4]

    seasons = xr.groupers.SeasonResampler(['DJF','MAM','JJA','SON'], drop_incomplete=False)

    # Find ozone concentration of historical dataset
    hist_flat = hist.mean(dim=['lat','lon','lev','ilev'])
    hist_seasonal = hist_flat.resample({'time':seasons}).mean()
    hist_o = hist_seasonal.O3.values

    # Find ozone concentration of other dataset
    other_flat = other.mean(dim=['lat','lon','lev','ilev'])
    other_seasonal = other_flat.resample({'time':seasons}).mean()
    other_o = other_seasonal.O3.values

    # Calculate percent contributions
    percents = []
    for i in range(len(hist_o)):
        true_val = hist_o[i]
        masked_val = other_o[i]
        per_cont = ((true_val-masked_val)/true_val) * 100
        percents.append(per_cont)

    # Plot figure
    fig = plt.figure(figsize=(10,8))
    ax = plt.axes()
    plt.plot(hist_seasonal.time, percents)

    locator = mdates.MonthLocator(bymonth=[12, 3, 6, 9], bymonthday=1)
    plt.gca().xaxis.set_major_locator(locator)
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.gca().tick_params(axis='x', labelrotation=45, labelsize=14)
    plt.xlabel('Time', fontsize=14)
    plt.ylabel('Percentage (%)', fontsize=14)
    plt.title(f'Percent Contribution of {reg_name} for Global Atmospheric Ozone Concentrations')

In [2]:
def sfc_ozone_percent_cont(hist, other, reg_name, time_range=None):
    # Handle dates
    try:
        len(hist['time'])==len(other['time'])
    except:
        raise KeyError('Both datasets must have the same length time coordinate')
        
    if time_range != None:
        start = str(time_range[0])
        end = str(time_range[1])
        hist = hist.sel(time=slice(start, end))
        other = other.sel(time=slice(start, end))
    else:
            start = str((hist['time'].values)[0])[:4]
            end = str((hist['time'].values)[len(hist['time'])-1])[:4]

    seasons = xr.groupers.SeasonResampler(['DJF','MAM','JJA','SON'], drop_incomplete=False)

    # Find surface ozone concentrations in historical dataset
    hist_flat = hist.mean(dim=['lat','lon','ilev'])
    hist_seasonal = hist_flat.resample({'time':seasons}).mean()
    hist_o = hist_seasonal.O3_SRF.values

    # Find surface ozone concentrations in masked dataset
    other_flat = other.mean(dim=['lat','lon','ilev'])
    other_seasonal = other_flat.resample({'time':seasons}).mean()
    other_o = other_seasonal.O3_SRF.values

    # Calculate percent contributions
    percents = []
    for i in range(len(hist_o)):
        true_val = hist_o[i]
        masked_val = other_o[i]
        per_cont = ((true_val-masked_val)/true_val) * 100
        percents.append(per_cont)

    # Plot figure
    fig = plt.figure(figsize=(10,8))
    ax = plt.axes()
    plt.plot(hist_seasonal.time, percents)

    locator = mdates.MonthLocator(bymonth=[12, 3, 6, 9], bymonthday=1)
    plt.gca().xaxis.set_major_locator(locator)
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.gca().tick_params(axis='x', labelrotation=45, labelsize=14)
    plt.legend(fontsize='large')
    plt.xlabel('Time', fontsize=14)
    plt.ylabel('Percentage (%)', fontsize=14)
    plt.title(f'Percent Contribution of {reg_name} for Global Surface Ozone Concentrations')

In [3]:
def all_sfc_ozone_pc(hist, aunz, bona, tena, table=False):
    seasons = xr.groupers.SeasonResampler(['DJF','MAM','JJA','SON'], drop_incomplete=False)
    keys = ['AUNZ', 'BONA', 'TENA']
    datalist = [aunz, bona, tena]
    ozone = {}
    times = []
    
    # Work with historical data first
    hist_flat = hist.mean(dim=['lat','lon','ilev'])
    hist_seasonal = hist_flat.resample({'time':seasons}).mean()
    hist_sfc = hist_seasonal.sel(lev = max(hist_seasonal.lev.values))
    hist_o = hist_sfc.O3.values
    times.append(hist_sfc.time.values)

    for r in range(3):
        data = datalist[r]
        data_flat = data.mean(dim=['lat','lon','ilev'])
        data_seasonal = data_flat.resample({'time':seasons}).mean()
        data_sfc = data_seasonal.sel(lev = max(data_seasonal.lev.values))
        model_o = data_sfc.O3.data
        key = keys[r]
        ozone[key] = model_o
         
    # Plot figure
    fig = plt.figure(figsize=(10,8))
    ax = plt.axes()
    
    labels = ['AUNZ', 'BONA', 'TENA']
    colors = ['orangered', 'saddlebrown', 'gold']

    percents_to_plot = []
    for k in keys:
        by_region = ozone[k]
        percents = []
        for n in range(len(by_region)):
            truth = hist_o[n]
            masked = by_region[n]
            contrib = ((truth-masked)/truth) * 100
            percents.append(contrib)
        percents_to_plot.append(percents)

    for x in range(len(datalist)):
        plotting_percents = percents_to_plot[x]
        plt.plot(times[0], plotting_percents, label = labels[x], c=colors[x])
        if table == True:
            print('{:>20}'.format(keys[x]))
            print('{:<20}{:>20}'.format('Date', 'Percent (%)'))
            for p in range(len(plotting_percents)):
                date = str(times[0][p])[:10]
                percentage = str((plotting_percents[p]).compute())
                print('{:<20}{:>20}'.format(date, percentage))
            print('\n')

    locator = mdates.MonthLocator(bymonth=[12, 3, 6, 9], bymonthday=1)
    plt.gca().xaxis.set_major_locator(locator)
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.gca().tick_params(axis='x', labelrotation=45, labelsize=14)
    
    plt.legend(fontsize='large')
    plt.xlabel('Time', fontsize=14)
    plt.ylabel('Percentage (%)', fontsize=14)
    plt.title('Percent Contribution by Region to Global Surface Ozone', fontsize=16)

In [4]:
def all_sfc_pm_pc(hist, aunz, bona, tena, table=False):
    seasons = xr.groupers.SeasonResampler(['DJF','MAM','JJA','SON'], drop_incomplete=False)
    keys = ['AUNZ', 'BONA', 'TENA']
    datalist = [aunz, bona, tena]
    pm = {}
    times = []
    
    # Work with historical data first
    hist_flat = hist.mean(dim=['lat','lon','ilev'])
    hist_seasonal = hist_flat.resample({'time':seasons}).mean()
    hist_sfc = hist_seasonal.sel(lev=max(hist_seasonal.lev.values))
    hist_pm = ((hist_sfc.PM25.values)*1e9)/1.225
    times.append(hist_seasonal.time.values)

    for r in range(3):
        data = datalist[r]
        data_flat = data.mean(dim=['lat','lon','ilev'])
        data_seasonal = data_flat.resample({'time':seasons}).mean()
        data_sfc = data_seasonal.sel(lev=max(data_seasonal.lev.values))
        model_pm = ((data_sfc.PM25.values)*1e9)/1.225
        key = keys[r]
        pm[key] = model_pm
         
    # Plot figure
    fig = plt.figure(figsize=(10,8))
    ax = plt.axes()
    
    labels = ['AUNZ', 'BONA', 'TENA']
    colors = ['orangered', 'saddlebrown', 'gold']

    percents_to_plot = []
    for k in keys:
        by_region = pm[k]
        percents = []
        for n in range(len(by_region)):
            truth = hist_pm[n]
            masked = by_region[n]
            contrib = ((truth-masked)/truth) * 100
            percents.append(contrib)
        percents_to_plot.append(percents)

    for x in range(len(datalist)):
        plotting_percents = percents_to_plot[x]
        plt.plot(times[0], percents_to_plot[x], label = labels[x], c=colors[x])
        if table == True:
            print('{:>20}'.format(keys[x]))
            print('{:<20}{:>20}'.format('Date', 'Percent (%)'))
            for p in range(len(plotting_percents)):
                print('{:<20}{:>20}'.format(str(times[0][p])[:10], plotting_percents[p]))
            print('\n')

    locator = mdates.MonthLocator(bymonth=[12, 3, 6, 9], bymonthday=1)
    plt.gca().xaxis.set_major_locator(locator)
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.gca().tick_params(axis='x', labelrotation=45, labelsize=14)
    
    plt.legend(fontsize='large')
    plt.xlabel('Time', fontsize=14)
    plt.ylabel('Percentage (%)', fontsize=14)
    plt.title('Percent Contribution by Region to Global Surface PM₂₅', fontsize=16)